# BW \#120 Pennies
Is the US going to abolish the penny?

The most recent annual report from the US Mint (https://www.usmint.gov/content/dam/usmint/reports/2024-annual-report.pdf) indicated that 80 percent of purchases are currently digital, meaning that even in the worst-case scenario, 20 percent of transactions might have to worry about pennies. That same report indicated that as of last year, each penny cost 3.69 cents to create, meaning that its face value was less than one third a penny's raw cost. Plus, they found that cash was only used in 20 percent of purchases, making it even less important to have physical pennies.

## Data and six questions
This week, our data will come from Numista (https://en.numista.com/), a site with numismatic (i.e., coin-related) information. The site is mainly aimed at coin collectors, including buyers and sellers. But it includes information about enough coins currently in use for our purposes.

## Challenges
The learning goals include API usage, JSON manipulation, strings, regular expressions and grouping 

- Assemble a list of dicts about the coins in Numista's db : using the API retrieve information about each coin that is in use in 2025. From that data, create a JSON file. The file should be in the form of a list of dicts (or an array of objects). You'll need to sign up for a free Numista account and then for a free API key at https://en.numista.com/api/api_key.php . You'll then need to use `requests` to ask for` https://api.numista.com/v3/types`, passing a `params` keyword argument with `{'category':'coin', 'q':'circulation', 'count':50, 'year':2025, 'lang':'en'}` as the value, and a `headers` keyword argument with `{'Numista-API-Key': api_key}` as the value, where `api_key` is set to your API key. Iterate, in a `for` loop, from `range(1,10)`, using `requests.get` to the URL you've constructed, adding the current number (from your iteration) to `params['page']`, so as to request the current page. The response will be in JSON, with the data itself available in the '`types`' key, a list of dicts. If the list has length 0, then exit from the loop. Otherwise, add that list of dicts to the list of all coins you've found, and go back for the next page.
- Create a JSON file based on the data you've retrieved: Iterate over the list of coin dicts you created. For each one, retrieve from `https://api.numista.com/v3/types/COIN_ID`, where `COIN_ID` is the '`id`' value from the current coin. You'll have to pass the same '`Numista-API-Key`' header as before. The only parameter you'll need to send is '`lang`', indicating '`en`' for English. Take each returned object, decode it from JSON, and append it to a list of dicts. In the end, use json.dump to write that list to a JSON file.
This post is for paying subscribers only
Subscribe now
Already have an account? Sign in

This API is typical, in that it uses a "REST" URL-based system. The idea is that you make an HTTP query describing the coins you want to learn about. You get back a series of responses, each with brief information about a coin matching the query, including a coin API. You can then use that coin ID to retrieve information about a particular coin.
- Parameters, which we set in a dict. These parameters describe the query that we want to make to the Numista server.
- Headers, which are sent as metadata with the HTTP request. In our case, there's a single header we need to set and send; the name is `'Numista-API-Key'`, and the value is the API key that you got from the Numista site

In [1]:
import pandas as pd
import requests

base_url = 'https://api.numista.com/v3/types'
params = {'category':'coin', 'q':'circulation', 'count':50, 'year':2025, 'lang':'en'}
headers = {'Numista-API-Key': 'FChonw7dYtFsauEw02vT0yda1OimN3k99HakohZk'}


The parameters describe the query we want to run. In this case, we're asking for coins, for the string '`circulation`' to appear in the coin's description, for the coin to be relevant and active in the year 2025, and for the results to be returned in English. We also indicate that we want to get a maximum of 25 records back with each request, although from what I can tell, that's already the default.

If we run this, we will get info back about 50 coins. But which 50? And what if there are more than 50? For that reason, we'll need to add one final parameter, `page`, which indicates the page of data we want to receive, starting with 1. If the response contains zero records, then we know that we've gone beyond the final page of records.

In [2]:
for page_number in range(1,10):
    print(f'Retreiving page {page_number}...')
    params['page'] = page_number # inside the loop to increment the page number each time
    response = requests.get(base_url , params = params, headers = headers)
    all_coin = []
    data = response.json()['types']
    if len(data) == 0:
        print(f'No data at page {page_number}; exiting')
        break

    print(f'\tGot {len(data)} records; adding and continuing...') # printing the retrieved data for each iteration

    all_coin += data

Retreiving page 1...
	Got 50 records; adding and continuing...
Retreiving page 2...
	Got 50 records; adding and continuing...
Retreiving page 3...
	Got 50 records; adding and continuing...
Retreiving page 4...
	Got 50 records; adding and continuing...
Retreiving page 5...
	Got 50 records; adding and continuing...
Retreiving page 6...
	Got 28 records; adding and continuing...
Retreiving page 7...
No data at page 7; exiting


#### Create a JSON file based on the data you've retrieved: Iterate over the list of coin dicts you created. For each one, retrieve from https://api.numista.com/v3/types/COIN_ID, where COIN_ID is the 'id' value from the current coin. You'll have to pass the same 'Numista-API-Key' header as before. The only parameter you'll need to send is 'lang', indicating 'en' for English. Take each returned object, decode it from JSON, and append it to a list of dicts. In the end, use json.dump to write that list to a JSON file

We have a list of dicts, and each dict gives the overview of a coin in the Numista database. But we need to get more details, which means making one additional API call for each coin.

In [3]:
import json
output = []
for index, one_coin in enumerate(all_coin,1): #use enumerate to get the coin id and the current number of coins starting at 1
    coin_id = one_coin['id']
    response = requests.get(f'{'https://api.numista.com/v3/types/COIN_ID'}/types/{coin_id}', params={'lang':'eng'}, headers = headers)
    if response.status_code == 200: # check if the request was successful
        coin_data = response.json()
        output.append(coin_data)

print(f'Writing {len(output)} records')
with open('coins.json', 'w') as f:
    json.dump(output, f)
print('Done.')

Writing 0 records
Done.
